#Code for training a CNN (resnet) to classify fruits and vegetables

In [1]:
#Import required packages
import kagglehub
import os
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
import torch.nn.utils.prune as prune
from torch.utils.data import DataLoader
from torchprofile import profile_macs

import json

In [2]:
#Hyperparmeters
batch_size = 10
learning_rate = 1e-4
num_epochs = 20
patience = 3 #Number of epochs to wait for improvement before stopping training
class_number = 51

In [3]:
#Helper functions (from assignment 1)
def get_model_macs(model, inputs) -> int:
    return profile_macs(model, inputs)

def get_num_parameters(model: nn.Module, count_nonzero_only=False) -> int:
    """
    calculate the total number of parameters of model
    :param count_nonzero_only: only count nonzero weights
    """
    num_counted_elements = 0
    for param in model.parameters():
        if count_nonzero_only:
            num_counted_elements += param.count_nonzero()
        else:
            num_counted_elements += param.numel()
    return num_counted_elements

def get_model_size(model: nn.Module, data_width=32, count_nonzero_only=False) -> int:
    """
    calculate the model size in bits
    :param data_width: #bits per element
    :param count_nonzero_only: only count nonzero weights
    """
    return get_num_parameters(model, count_nonzero_only) * data_width

Byte = 8
KiB = 1024 * Byte
MiB = 1024 * KiB
GiB = 1024 * MiB

In [4]:
# Download latest version
path = kagglehub.dataset_download("evan15h/cooking-ingredients-dataset")

print("Path to dataset files:", path)

for root, dirs, files in os.walk(path):
    print(root)
    break

Path to dataset files: /home/arberube/.cache/kagglehub/datasets/sunnyagarwal427444/food-ingredient-dataset-51/versions/1
/home/arberube/.cache/kagglehub/datasets/sunnyagarwal427444/food-ingredient-dataset-51/versions/1


In [5]:
#Load the Dataset

#train_dataset = datasets.ImageFolder(root=path+"/huggingface/Train")
#val_dataset = datasets.ImageFolder(root=path+"/huggingface/val")
#train_dataset = datasets.ImageFolder(root=path+"/hf_selected_ingredients")
#val_dataset = datasets.ImageFolder(root=path+"/hf_selected_ingredients")

#Save ingredients labels
ingredients = train_dataset.classes              #list of class names in index order
ingredients_to_idx = train_dataset.class_to_idx    # dict name -> index

# Build mappings
id2label = {str(i): name for i, name in enumerate(ingredients)}
label2id = {name: str(i) for i, name in enumerate(ingredients)}

with open("../JSON/id2label_produce.json", "w") as f:
    json.dump(id2label, f, indent=2)

with open("../JSON/label2id_produce.json", "w") as f:
    json.dump(label2id, f, indent=2)

print("Number of classes:", len(train_dataset.classes))
print("Class names:", train_dataset.classes)

Number of classes: 51
Class names: ['Amaranth', 'Apple', 'Banana', 'Beetroot', 'Bell pepper', 'Bitter Gourd', 'Blueberry', 'Bottle Gourd', 'Broccoli', 'Cabbage', 'Cantaloupe', 'Capsicum', 'Carrot', 'Cauliflower', 'Chilli pepper', 'Coconut', 'Corn', 'Cucumber', 'Dragon_fruit', 'Eggplant', 'Fig', 'Garlic', 'Ginger', 'Grapes', 'Jalepeno', 'Kiwi', 'Lemon', 'Mango', 'Okra', 'Onion', 'Orange', 'Paprika', 'Pear', 'Peas', 'Pineapple', 'Pomegranate', 'Potato', 'Pumpkin', 'Raddish', 'Raspberry', 'Ridge Gourd', 'Soy beans', 'Spinach', 'Spiny Gourd', 'Sponge Gourd', 'Strawberry', 'Sweetcorn', 'Sweetpotato', 'Tomato', 'Turnip', 'Watermelon']


In [6]:
mean=[0.485, 0.456, 0.406]
std=[0.229, 0.224, 0.225]

#Apply transform to the dataset
transform = transforms.Compose([
    transforms.Resize((224, 224)),   # ResNet standard
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],  # ImageNet mean
        std=[0.229, 0.224, 0.225]    # ImageNet std
    )
])

#save mean and std before normalization
norm_stats = {
    "mean": mean,
    "std": std
}
with open("../JSON/norm_stats_produce.json", "w") as f:
    json.dump(norm_stats, f)

train_dataset.transform = transform #set the transform attribute of the ImageFolder class
val_dataset.transform = transform

#Create dataloaders
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=batch_size, shuffle=False)

In [7]:
#Load Resnet model
model = models.resnet18(pretrained=True)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, len(train_dataset.classes)) #make the last fc layer output the correct number of classes

#Setup GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

/home/arberube/ODDL/labEnv/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/arberube/ODDL/labEnv/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

***Training*** This code trains the model while gradually prunning out the least important weightsuntil it reaches a target sparsity of 60% or is stopped. It looks at performance using early stopping and  saves the most accurate version of the  network.

In [ ]:
def get_sparsity(model):
    total_zeros = 0
    total_elements = 0
    for module in model.modules():
        if isinstance(module, nn.Conv2d) or isinstance(module, nn.Linear):
            weight = getattr(module, 'weight', None)
            if weight is not None:
                total_zeros += float(torch.sum(weight == 0))
                total_elements += float(weight.nelement())
    return total_zeros / total_elements if total_elements > 0 else 0.0

def apply_global_pruning(model, amount):
    parameters_to_prune = []
    for module in model.modules():
        if isinstance(module, nn.Conv2d) or isinstance(module, nn.Linear):
            parameters_to_prune.append((module, 'weight'))

    prune.global_unstructured(
        parameters_to_prune,
        pruning_method=prune.L1Unstructured,
        amount=amount,
    )

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

best_loss = float('inf')
epochs_without_improvement = 0

prune_start_epoch = 5
prune_interval = 2
prune_amount_per_step = 0.1
current_sparsity = 0.0
max_sparsity = 0.6

history_epochs = []
history_loss = []
history_acc = []
history_sparsity = []

for epoch in range(num_epochs):
    print(f"\n--- Starting Epoch {epoch} ---")

    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    total_steps = len(train_loader)

    for i, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

# Print epoch progress every 50 batches
#(Put this in becuase it is running really slow and I want to make sure it hasn't crashed)
        if (i + 1) % 50 == 0 or (i + 1) == total_steps:
            print(f"  Epoch [{epoch}/{num_epochs}] Step [{i+1}/{total_steps}] Loss: {loss.item():.4f}")

    epoch_loss = running_loss / total_steps
    epoch_acc = 100 * correct / total
    actual_sparsity = get_sparsity(model)

# Store metrics for graphing
    history_epochs.append(epoch)
    history_loss.append(epoch_loss)
    history_acc.append(epoch_acc)
    history_sparsity.append(actual_sparsity)

    print(f"End of Epoch {epoch} | Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2f}% | Actual Sparsity: {actual_sparsity:.4f}")


# Apply pruning gradually
    if epoch >= prune_start_epoch and epoch % prune_interval == 0:
        if current_sparsity < max_sparsity:
            current_sparsity += prune_amount_per_step
            print(f"  -> Applying pruning step... Target Sparsity: {current_sparsity:.2f}")
            apply_global_pruning(model, prune_amount_per_step)

# Early stopping
    if epoch_loss < best_loss:
        best_loss = epoch_loss
        epochs_without_improvement = 0
        torch.save(model.state_dict(), "best_model.pth")
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= patience:
        print("\nEarly stopping triggered. Halting training.")
        break

# Finalize Pruning & Save
print("\nFinalizing pruning and saving the model...")
for module in model.modules():
    if isinstance(module, nn.Conv2d) or isinstance(module, nn.Linear):
        try:
            prune.remove(module, 'weight')
        except:
            pass

torch.save(model.state_dict(), "resnet18_pruned.pth")
print("Done!")


--- Starting Epoch 0 ---


/home/arberube/ODDL/labEnv/lib/python3.13/site-packages/PIL/Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


  Epoch [0/20] Step [50/399] Loss: 2.6828
  Epoch [0/20] Step [100/399] Loss: 2.2373
  Epoch [0/20] Step [150/399] Loss: 0.9501
  Epoch [0/20] Step [200/399] Loss: 1.3116
  Epoch [0/20] Step [250/399] Loss: 1.8360
  Epoch [0/20] Step [300/399] Loss: 1.6453
  Epoch [0/20] Step [350/399] Loss: 0.8514
  Epoch [0/20] Step [399/399] Loss: 0.8118
End of Epoch 0 | Loss: 1.6094 | Accuracy: 63.43% | Actual Sparsity: 0.0000

--- Starting Epoch 1 ---
  Epoch [1/20] Step [50/399] Loss: 0.8078
  Epoch [1/20] Step [100/399] Loss: 0.9973
  Epoch [1/20] Step [150/399] Loss: 0.4578
  Epoch [1/20] Step [200/399] Loss: 0.4504
  Epoch [1/20] Step [250/399] Loss: 0.6922
  Epoch [1/20] Step [300/399] Loss: 0.6782
  Epoch [1/20] Step [350/399] Loss: 0.7359
  Epoch [1/20] Step [399/399] Loss: 0.3650
End of Epoch 1 | Loss: 0.5383 | Accuracy: 87.24% | Actual Sparsity: 0.0000

--- Starting Epoch 2 ---
  Epoch [2/20] Step [50/399] Loss: 0.1108
  Epoch [2/20] Step [100/399] Loss: 0.1041


***Plots of progress of training***

In [ ]:
import matplotlib.pyplot as plt


plt.figure(figsize=(14, 5))

# Plot 1: Loss & Accuracy
ax1 = plt.subplot(1, 2, 1)
ax1.plot(history_epochs, history_loss, label='Loss', color='red')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss', color='red')
ax1.tick_params(axis='y', labelcolor='red')

ax2 = ax1.twinx()  # Create a second y-axis for accuracy
ax2.plot(history_epochs, history_acc, label='Accuracy', color='blue')
ax2.set_ylabel('Accuracy (%)', color='blue')
ax2.tick_params(axis='y', labelcolor='blue')

plt.title('Training Loss and Accuracy')
plt.grid(True, alpha=0.3)

# Plot 2: Sparsity Over Time
plt.subplot(1, 2, 2)
plt.plot(history_epochs, history_sparsity, label='Sparsity', color='green', marker='o', linestyle='-')
plt.xlabel('Epochs')
plt.ylabel('Sparsity (Fraction of Zeros)')
plt.title('Model Sparsity Schedule')
plt.axhline(y=max_sparsity, color='gray', linestyle='--', label='Target Max Sparsity')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

***Quantization of the Network***

In [ ]:
from collections import namedtuple
from fast_pytorch_kmeans import KMeans

Codebook = namedtuple('Codebook', ['centroids', 'labels'])

def k_means_quantize(fp32_tensor: torch.Tensor, bitwidth=4, codebook=None):
    """
    Quantize tensor using k-means clustering.
    """
    if codebook is None:
        n_clusters = (2 ** bitwidth) -1 # leave room for zeros 
        kmeans = KMeans(n_clusters=n_clusters, mode='euclidean', verbose=0)
        labels = kmeans.fit_predict(fp32_tensor.reshape(-1, 1)).to(torch.long)
        centroids = kmeans.centroids.to(torch.float).view(-1)
        codebook = Codebook(centroids, labels)
        
    quantized_tensor = codebook.centroids[codebook.labels]
    
    fp32_tensor.copy_(quantized_tensor.view_as(fp32_tensor))
    
    return codebook

***apply_model_quantization*** Changed to avoid removing the changes to sparsity. updated to fix this by memorizing where the zeros are before quantization and putting them back to 0.0 after.

In [ ]:
def apply_model_quantization(model, bitwidth=4):
    """
    Iterates through the model and applies k-means quantization to Conv2d and Linear layers.
    Skips sensitive layers and protects pruned weights.
    """
    model_codebooks = {}
    
    print(f"Starting {bitwidth}-bit K-Means Quantization...")
    
    with torch.no_grad():
        for name, module in model.named_modules():
            
            # skip the highly sensitive first and last layers
            if name in ['conv1', 'fc']:
                print(f"  -> Skipping sensitive layer: {name}")
                continue
                
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                if hasattr(module, 'weight') and module.weight is not None:
                    print(f"  -> Quantizing layer: {name}")
                    
                    # Remember where the pruned zeros are ---
                    zero_mask = (module.weight.data == 0)
                    
                    # Apply k-means function to the weight data
                    codebook = k_means_quantize(module.weight.data, bitwidth=bitwidth)
                    
                    #Force the pruned weights back to  0.0 
                    module.weight.data[zero_mask] = 0.0
                    
                    model_codebooks[name] = codebook
                    
    print("Quantization complete!")
    return model_codebooks

***Actual Quantization***

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

# Load the Pruned Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#  ResNet18
model = models.resnet18()


# Adjust the final layer to match dataset
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, class_number) 

# Load the pruned weights saved from training
model.load_state_dict(torch.load("resnet18_pruned.pth", weights_only=True))
model = model.to(device)


# Apply K-Means Quantization
bitwidth = 4
codebooks = apply_model_quantization(model, bitwidth=bitwidth)


# Verification 
layer_to_check = 'layer1.0.conv1'

# Grab the specific layer dynamically
modules_dict = dict(model.named_modules())
weights = modules_dict[layer_to_check].weight.data

# Count unique values
unique_weights = torch.unique(weights)
expected_clusters = 2 ** bitwidth

print("\n--- Verification ---")
print(f"Checking layer: {layer_to_check}")

# 17 unique values (16 centroids + 1 pure zero).
print(f"Expected unique weights: {expected_clusters}")
print(f"Actual unique weight values: {len(unique_weights)}")

if len(unique_weights) in [expected_clusters, expected_clusters + 1]:
    print("Success")
else:
    print("Warning: Unique values do not match expected clusters.")

***Plot showing Number and distribution of weights pre and post quantization***

In [ ]:
import matplotlib.pyplot as plt
import torchvision.models as models
import torch
import torch.nn as nn
import numpy as np 


# Setup Models & Extract Weights
layer_name = 'layer1.0.conv1'

# Grab the Quantized Weights from the current model in memory
modules_dict = dict(model.named_modules())
quantized_weights = modules_dict[layer_name].weight.data.cpu().numpy().flatten()

# Create FP32 baseline
original_model = models.resnet18() 
num_ftrs = original_model.fc.in_features
original_model.fc = nn.Linear(num_ftrs, 51) 

# Load the baseline pruned weights
original_model.load_state_dict(torch.load("resnet18_pruned.pth", weights_only=True))
orig_modules_dict = dict(original_model.named_modules())
original_weights = orig_modules_dict[layer_name].weight.data.cpu().numpy().flatten()


# Generate the Comparison Graphs
plt.figure(figsize=(15, 6))

# Subplot 1: Original FP32 Distribution
plt.subplot(1, 2, 1)
plt.hist(original_weights, bins=150, color='royalblue', alpha=0.8, edgecolor='black', linewidth=0.2)
plt.title(f'Before: {layer_name} Pruned Weights (FP32)', fontsize=14)
plt.xlabel('Weight Value', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.grid(axis='y', alpha=0.3, linestyle='--')

#  Unique Weights count for FP32
orig_unique = len(np.unique(original_weights))
plt.text(0.95, 0.95, f"Unique Weights: {orig_unique}", transform=plt.gca().transAxes, 
         fontsize=11, verticalalignment='top', horizontalalignment='right',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Subplot 2: Quantized Distribution
plt.subplot(1, 2, 2)
plt.hist(quantized_weights, bins=150, color='crimson', alpha=0.8, edgecolor='black', linewidth=0.2)
plt.title(f'After: {layer_name} Quantized (4-bit K-Means)', fontsize=14)
plt.xlabel('Weight Value (Centroids)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.grid(axis='y', alpha=0.3, linestyle='--')

#  Unique Weights count for Quantized
quant_unique = len(np.unique(quantized_weights))
plt.text(0.95, 0.95, f"Unique Weights: {quant_unique}", transform=plt.gca().transAxes, 
         fontsize=11, verticalalignment='top', horizontalalignment='right',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show()

***Post Quant Training*** 

In [ ]:
import torch
import torch.nn as nn

def enforce_cluster_gradients_and_sparsity(model, codebooks):
    with torch.no_grad():
        for name, module in model.named_modules():
            if name in codebooks and hasattr(module, 'weight') and module.weight.grad is not None:
                codebook = codebooks[name]
                labels = codebook.labels.view_as(module.weight)
                
                zero_mask = (module.weight == 0)
                
                for cluster_idx in range(len(codebook.centroids)):
                    cluster_mask = (labels == cluster_idx)
                    
                    if cluster_mask.sum() > 0:
                        cluster_grad_mean = module.weight.grad[cluster_mask].mean()
                        module.weight.grad[cluster_mask] = cluster_grad_mean
                
                module.weight.grad[zero_mask] = 0.0

fine_tune_lr = 1e-4 
optimizer_ft = torch.optim.Adam(model.parameters(), lr=fine_tune_lr)
criterion = nn.CrossEntropyLoss()
fine_tune_epochs = 3 

print("Starting Post-Quantization Fine-Tuning")

for epoch in range(fine_tune_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for i, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)

        optimizer_ft.zero_grad()
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()

        enforce_cluster_gradients_and_sparsity(model, codebooks)
        optimizer_ft.step()
        
        with torch.no_grad():
            for name, module in model.named_modules():
                if name in codebooks and hasattr(module, 'weight'):
                    zero_mask = (module.weight == 0)
                    module.weight[zero_mask] = 0.0

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total
    
    print(f"Fine-Tune Epoch [{epoch+1}/{fine_tune_epochs}] | Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2f}%")

print("\nFine-tuning complete!")
print("\n--- Verifying Quantization Survival ---")

layer_to_check = 'layer1.0.conv1'
modules_dict = dict(model.named_modules())
final_unique_vals = len(torch.unique(modules_dict[layer_to_check].weight.data))

print(f"Checking layer: {layer_to_check}")
print(f"Unique weight values after training: {final_unique_vals} ")

if final_unique_vals <= 17:
    print("Success")
else:
    print("Warning: Clusters broke apart during training.")

torch.save(model.state_dict(), "resnet18_pruned_quantized_finetuned.pth")
print("Saved final model to 'resnet18_pruned_quantized_finetuned.pth'")

***Quant Plots*** Makes a nice comparsion graph

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision.models as models
import os


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def get_model_size(model, bitwidth=32):
    total_params = sum(p.numel() for p in model.parameters())
    # Size in bytes = total parameters * (bits per param / 8 bits per byte)
    size_bytes = total_params * (bitwidth / 8)
    return size_bytes / (1024**2) # Convert to MB

def evaluate_and_size(model_path, is_quantized=False):
    # Build architecture
    eval_model = models.resnet18()
    eval_model.fc = nn.Linear(eval_model.fc.in_features, 51) 
    
    # Load weights
    eval_model.load_state_dict(torch.load(model_path, weights_only=True))
    eval_model = eval_model.to(device)
    eval_model.eval()
    
    # Calculate Accuracy
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = eval_model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    acc = round(100 * correct / total, 2)
    
    # Calculate Size 
    bitwidth = 4 if is_quantized else 32
    size_mb = get_model_size(eval_model, bitwidth=bitwidth)
    
    return acc, round(size_mb, 2)

# ---------------------------------------------------------
# Metrics Collection
# ---------------------------------------------------------
print("Processing Baseline (32-bit) Model...")
original_acc, original_size = evaluate_and_size("resnet18_pruned.pth", is_quantized=False)

print("Processing Quantized (4-bit) Model...")
quantized_acc, quantized_size = evaluate_and_size("resnet18_pruned_quantized_finetuned.pth", is_quantized=True)


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
labels = ['Before\n(32-bit FP)', 'After\n(4-bit Quantized)']

# --- Plot 1: Accuracy Comparison ---
accuracies = [original_acc, quantized_acc]
bars1 = ax1.bar(labels, accuracies, color=['royalblue', 'forestgreen'], edgecolor='black', alpha=0.8, width=0.5)
ax1.set_ylabel('Accuracy (%)')
ax1.set_title('Accuracy Preservation', fontweight='bold')
ax1.set_ylim(0, 110)

for bar in bars1:
    yval = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, yval + 2, f"{yval}%", ha='center', fontweight='bold')

# --- Plot 2: Actual Model Size ---
sizes = [original_size, quantized_size]
bars2 = ax2.bar(labels, sizes, color=['slategray', 'crimson'], edgecolor='black', alpha=0.8, width=0.5)
ax2.set_ylabel('Memory Size (MB)')
ax2.set_title('Actual Model Size Comparison', fontweight='bold')
ax2.set_ylim(0, max(sizes) * 1.2)

for bar in bars2:
    yval = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, yval + 1, f"{yval} MB", ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

***Save & Test Final Model*** saves as produce_net.pth

In [ ]:
#Model Save
PATH = './produce_net.pth'

# Save the state_dict for better compatibility and weight tracking
torch.save(model.state_dict(), PATH)

print(f"Model weights saved to {PATH}")

In [ ]:
#Load model
PATH = './produce_net.pth'
model = torch.load(PATH, weights_only=False)

In [ ]:
#Test Model

model = models.resnet18()
num_ftrs = model.fc.in_features
model.fc = torch.nn.Linear(num_ftrs, 51) 

PATH = './produce_net.pth'
model.load_state_dict(torch.load(PATH, weights_only=True))

#Evaluate
model.to(device)
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

val_acc = 100 * (correct / total)
print(f"Validation accuracy: {val_acc:.2f} %")

In [ ]:
#Get model information
model_size = get_model_size(model)
print(f"Model Size: {model_size:.2f} MB")

num_parameters = get_num_parameters(model)
print(f"Parameter Count (M) {num_parameters/1e6:.2f}")

dummy_input = torch.randn(1, 3, 224, 224).to(device)
macs = get_model_macs(model, dummy_input)
print(f"MACS (B) {macs/1e9:.2f}")

***End of Project***`